In [1]:
import pandas as pd
import gspread
from google.oauth2.service_account import Credentials

In [2]:
from datetime import datetime as dt
from datetime import timedelta

Настройка прав доступа и авторизация по сервисному аккаунту Google


In [3]:
scopes = [
    'https://www.googleapis.com/auth/spreadsheets',
    'https://www.googleapis.com/auth/drive'
]

Авторизация и создание клиента

In [4]:
creds = Credentials.from_service_account_file('credentials.json', scopes=scopes)
client = gspread.authorize(creds)

Получаем вчерашнюю дату

In [5]:
yesterday = dt.today() - timedelta(days=1)
months_dict = {
    1: "Январь",
    2: "Февраль",
    3: "Март",
    4: "Апрель",
    5: "Май",
    6: "Июнь",
    7: "Июль",
    8: "Август",
    9: "Сентябрь",
    10: "Октябрь",
    11: "Ноябрь",
    12: "Декабрь"
}
year = yesterday.year
month = months_dict.get(yesterday.month)
day = yesterday.day



print(f'Вчерашняя дата: \n Год: {year} \n Месяц: {month} \n Число: {day}')

Вчерашняя дата: 
 Год: 2026 
 Месяц: Сентябрь 
 Число: 8


In [6]:
full_date  = yesterday.strftime('%d.%m.%Y')
full_date

'08.09.2026'

Окрываем таблицу и подклчаемся к необходимому листу

In [7]:
sheet = client.open_by_key('1o1mIcsXQht1NFhgsq7CMKI3derC8xOSrRgGC9GYu144').worksheet(f'{month} {year}')

Приводим таблицу в необходимый вид с названиями столбцов и индексами в виде дат.

In [8]:
data = sheet.get_all_values()
df = pd.DataFrame(data)
df = df.loc[3:]
df_basic = df.copy()
df_basic.columns = df.iloc[0]


In [9]:
basic_cols = df.iloc[0][:24].to_list()
basic_cols.pop(1)


'Дата'

In [10]:
df_basic = df_basic[1:].reset_index(drop=True)
df_basic = df_basic.set_index('Дата')

df_basic = df_basic.iloc[:, :24]

In [ ]:
df_basic.columns

Index(['номер недели', 'Трафик', 'Уникальные', 'vs LY', 'vs LY %',
       'Переход в каталог', 'CTR, %', 'Положил в корзину', 'CTR, %',
       'Оформил заказ', 'CTR, %', 'План Руб', 'План Заказы', 'Заказы Сайт, шт',
       'Сумма заказов РУБ', 'Средний чек закза', 'CTR, %',
       'Кол-во проданных ед', 'ШТ', 'vs LY', 'diff', 'Сертификаты, шт',
       'Сертификаты, Руб', 'LFL'],
      dtype='object', name=3)

Разобъем наш датасет на отдельные таблицы, разные потоки информации(яндекс метрики, сайт, а так же вручную вносимая инфа о регистрациях и проданных сертификатах()заносящиеся позже).

In [25]:
df_basic_yandex = df_basic.iloc[:, :10][['номер недели', 'Трафик', 'Уникальные', 'vs LY', 'Переход в каталог', 'Положил в корзину', 'Оформил заказ']]
df_basic_pred = df_basic[['номер недели', 'План Руб', 'План Заказы', 'Заказы Сайт, шт', 'Сумма заказов РУБ']]
df_basic_registration = df_basic.iloc[:, 18:21][[ 'ШТ', 'vs LY']]
df_basic_certif = df_basic[['номер недели','Сертификаты, шт', 'Сертификаты, Руб']]

In [28]:
df_basic_certif

3,номер недели,"Сертификаты, шт","Сертификаты, Руб"
Дата,,,
01.09.2026,36,,
02.09.2026,36,,
03.09.2026,36,,
04.09.2026,36,,
05.09.2026,36,,
06.09.2026,37,,
07.09.2026,37,,
08.09.2026,37,,
09.09.2026,37,,
